# TAL-59 — Verifica critica del fix a `riapertura_dopo_revoca`

**Obiettivo di questo notebook:** non limitarsi a confermare che il fix "funziona", ma metterlo sotto pressione — mostrare cosa toglie, cosa lascia, cosa *perde* (falsi negativi), e se generalizza oltre i casi che abbiamo già letto a mano durante lo sviluppo (rischio di overfitting sui casi trovati).

Tutte le query girano **dal vivo su `talia.db` reale** (161MB, DB di produzione locale), confrontando il codice **prima del fix** (commit `f26e94d`, `main`) e **dopo il fix** (commit `3e459e4`, branch `feat/TAL-59-fix-riapertura-falsi-positivi`) sullo stesso identico stato del database — quindi ogni differenza nei risultati è dovuta solo al codice, non a dati diversi.

**Card di riferimento:** [`docs/cards/TAL-59.md`](../../docs/cards/TAL-59.md).

In [1]:
"""Setup: carica il codice PRIMA del fix (f26e94d, main) e DOPO il fix (3e459e4,
questo branch) come due moduli separati, entrambi puntati sullo stesso talia.db.
"""
import importlib.util
import sqlite3
import subprocess
import sys
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

REPO = Path("/Users/dom/Documents/GitHub/talia")
DB = REPO / "talia.db"
PRE_FIX_COMMIT = "f26e94d"  # main, prima di TAL-59
POST_FIX_COMMIT = "3e459e4"  # branch feat/TAL-59-..., primo giro del fix (Tentativo 1-3)

sys.path.insert(0, str(REPO / "src"))


def carica_modulo_da_commit(commit: str, path_relativo: str, nome_modulo: str):
    """Estrae un file da un commit specifico via `git show` e lo carica come
    modulo Python isolato (senza toccare il working tree)."""
    sorgente = subprocess.run(
        ["git", "show", f"{commit}:{path_relativo}"],
        cwd=REPO, capture_output=True, text=True, check=True,
    ).stdout
    tmp_path = Path(f"/tmp/_tal59_{nome_modulo}_{commit}.py")
    tmp_path.write_text(sorgente)
    spec = importlib.util.spec_from_file_location(nome_modulo, tmp_path)
    modulo = importlib.util.module_from_spec(spec)
    sys.modules[nome_modulo] = modulo
    spec.loader.exec_module(modulo)
    return modulo


# Codice ORIGINALE (pre-TAL-59)
riap_pre = carica_modulo_da_commit(
    PRE_FIX_COMMIT,
    "src/talia/modulo2_scraping/red_flags/riapertura_revoca.py",
    "riap_pre",
)

# Codice ATTUALE (working tree, dopo l'ultimo giro di correzioni) — import normale
from talia.modulo2_scraping.red_flags import riapertura_revoca as riap_post
from talia.engine import catena as catena_post


def nuova_connessione():
    con = sqlite3.connect(DB)
    con.row_factory = sqlite3.Row
    return con


nomi_enti = {r[0]: r[1] for r in nuova_connessione().execute("SELECT id, denominazione FROM enti")}
print(f"DB: {DB}  ({DB.stat().st_size / 1e6:.0f} MB)")
print(f"Pre-fix module:  {riap_pre.__file__}")
print(f"Post-fix module: {riap_post.__file__}")

DB: /Users/dom/Documents/GitHub/talia/talia.db  (161 MB)
Pre-fix module:  /tmp/_tal59_riap_pre_f26e94d.py
Post-fix module: /Users/dom/Documents/GitHub/talia/src/talia/modulo2_scraping/red_flags/riapertura_revoca.py


## Prova 1 — Riduzione grezza dei flag

Quanti flag `riapertura_dopo_revoca` produce il codice prima e dopo, sullo stesso DB? Questo è il numero che avevo già dato a voce (82 → 23 → 10); qui lo ricalcolo dal vivo per verificarlo, non fidandomi del numero riportato in chat.

In [2]:
ris_pre = riap_pre.rileva_riapertura_dopo_revoca(nuova_connessione())
ris_post = riap_post.rileva_riapertura_dopo_revoca(nuova_connessione())

proc_pre = {r.procedimento_revocato_id for r in ris_pre}
proc_post = {r.procedimento_revocato_id for r in ris_post}

riepilogo = pd.DataFrame([
    {"versione": "PRIMA del fix (f26e94d)", "righe_flag": len(ris_pre), "procedimenti_distinti": len(proc_pre)},
    {"versione": "DOPO il fix (working tree)", "righe_flag": len(ris_post), "procedimenti_distinti": len(proc_post)},
])
riepilogo["riduzione_%"] = None
riepilogo.loc[1, "riduzione_%"] = round(100 * (1 - len(ris_post) / len(ris_pre)), 1)
riepilogo

,versione,righe_flag,procedimenti_distinti,riduzione_%
0,PRIMA del fix (f26e94d),82,67,None
1,DOPO il fix (working tree),10,8,87.8


**Risultato:** 82→10 righe di flag, **67→8 procedimenti distinti** (-88,1%). Il numero di procedimenti distinti è il dato che conta di più (le righe duplicano quando lo stesso procedimento matcha più di un atto successivo) — ed è più severo di quanto avessi detto a voce prima (avevo in mente un sottoinsieme di enti, non l'immagine completa). Andiamo a vedere *chi* è stato tolto, non solo *quanti*.

## Prova 2 — Cosa è stato tolto, ed è corretto toglierlo?

Elenco completo (non un campione) dei procedimenti che PRIMA generavano un flag e ora non più. Per ciascuno leggo l'oggetto pieno e do un giudizio esplicito — così chi legge può controllare il mio giudizio, non solo il numero aggregato.

In [3]:
con = nuova_connessione()

tolti = sorted(proc_pre - proc_post)
righe = []
for pid in tolti:
    p = con.execute("SELECT ente_id, oggetto, stato_finale FROM procedimenti WHERE id=?", (pid,)).fetchone()
    righe.append({
        "proc_id": pid,
        "ente": nomi_enti.get(p["ente_id"], "?"),
        "stato": p["stato_finale"],
        "oggetto": p["oggetto"],
    })
df_tolti = pd.DataFrame(righe)
print(f"Procedimenti rimossi dal fix: {len(df_tolti)}")
df_tolti

Procedimenti rimossi dal fix: 59


,proc_id,ente,stato,oggetto
0,897,Comune di Caltanissetta,annullato,T.A.R. - PERSONA (1) IN ATTI MEGLIO SPECIFICATA C/COMUNE - RICORSO PER VIOLAZIONE DEGLI ARTT. 22 E SEG. L. 241/90 - ...
1,6469,Comune di Custonaci,revocato,APPROVAZIONE RELAZIONE A CONSUNTIVO SULLA PERFORMANCE – ANNO 2024
2,7092,Comune di Bompietro,annullato,CENSIMENTODELLE AUTOVETTURE DI SERVIZIO 2026
3,7532,Comune di Taormina,annullato,"IMPEGNO DI SPESA E LIQUIDAZIONE PER CANONE DI LOCAZIONE DAL 15.03.2023 AL 30.06.2023 PER L'IMMOBILE, SITO IN VIA FRA..."
4,7560,Comune di Taormina,annullato,"DETERMINA DI IMPEGNO IN FAVORE DELLA F.I.G.C. - LND SETTORE DEROGHE ED OMOLOGAZIONI, PER LA QUOTA DA VERSARE AI FINI..."
5,8104,Comune di Catania,revocato,IRROGAZIONE DELLA SANZIONE AMMINISTRATIVA PECUNIARIA AI SENSI DELL'ART. 37 DEL D.P.R. 380/2001 PER INTERVENTI REALIZ...
6,8117,Comune di Catania,revocato,TRASMISSIONE VERB.N.200579/26 ART.193/2 CDS E RELATIVI ALLEGATI REDATTI DALLA POL.LOCALE DI ACIREALE PER LA PUBBLICA...
7,8163,Comune di Giarre,revocato,INGEGNERE GIUSEPPA RITA LEONARDI C/ COMUNE DI GIARRE. COMPONIMENTO BONARIO DELLA CONTROVERSIA. INVITO ALLA NEGOZIAZI...
8,8203,Comune di Giarre,revocato,CHIUSURA TEMPORANEA AL TRANSITO VEICOLARE DI VIA ETNA DALL'INTERSEZIONE CON VIA FILIPPO MEDA PER LAVORI DI MANUTENZI...
9,8214,Comune di Giarre,annullato,"CETTINA PATANE', GRASSO DAVIDE, SANTONOCETO ROSALBA, LA SPINA MARIA ALFINA E GIUFFRIDA MARCELLO C/ COMUNE DI GIARRE...."


**Lettura riga per riga dei 59 tolti** (categorie, non un campione — li ho letti tutti):

- **17 casi da un solo ente**, Sant'Agata li Battiati: regolamenti comunali, adesioni societarie, piani di rientro dal disavanzo, patrocini gratuiti, elenchi di delibere di giunta — nessuno è un bando. È il segnale più forte trovato in questa verifica: prima del fix, quasi un quarto di tutti i (falsi) flag veniva da un singolo ente con un pattern amministrativo che il matching Jaccard scambiava sistematicamente per "bandi ripubblicati". Non l'avevo notato analizzando solo i 9 campioni originari di TAL-12 (nessuno di quei 9 era di Sant'Agata).
- **14 casi da Giarre**: contenziosi (TAR, appelli, liquidazioni onorari legali), deleghe assessori, ordinanze — la stessa famiglia già documentata in TAL-59 Tentativo 2/3.
- **Il resto**: sanzioni edilizie, verbali di infrazione al codice della strada, incarichi legali, revoche di assessori, piani di rientro, censimenti veicoli, certificazioni. Il caso più illustrativo per assurdità è **Pedara 10663**: il "procedimento revocato" e il suo "riaperto" erano — testualmente — due pubblicazioni entrambe intitolate `CERTIFICAZIONE COVID`, evidentemente non un bando in nessun senso del termine.

**Nessuno dei 59 letti mi sembra un caso dubbio da riconsiderare** — la lettura manuale conferma la correttezza dell'esclusione riga per riga, non solo in aggregato.

## Prova 3 — Cosa resta: sono davvero tutti casi validi?

I procedimenti che *ancora oggi* generano il flag, con oggetto revocato + oggetto di riapertura completi. Rileggo ciascuno per verificare che sia plausibilmente un vero "bando revocato e ripubblicato", non solo che passi il filtro di dominio.

In [4]:
visti = set()
righe = []
for r in ris_post:
    key = (r.ente_id, r.procedimento_revocato_id)
    if key in visti:
        continue
    visti.add(key)
    righe.append({
        "ente": nomi_enti.get(r.ente_id, "?"),
        "proc_id": r.procedimento_revocato_id,
        "giorni": r.giorni_tra_revoca_e_riapertura,
        "jaccard": round(r.similarita_jaccard, 2),
        "oggetto_revocato": r.oggetto_revocato,
        "oggetto_riapertura": r.oggetto_riapertura,
    })
df_finali = pd.DataFrame(righe).sort_values("proc_id").reset_index(drop=True)
print(f"Procedimenti che generano ancora un flag: {len(df_finali)}")
df_finali

Procedimenti che generano ancora un flag: 8


,ente,proc_id,giorni,jaccard,oggetto_revocato,oggetto_riapertura
0,Comune di Palma di Montechiaro,692,240,0.76,"DETERMINA A CONTRARRE E AFFIDAMENTO DIRETTO, AI SENSI DELL’ART. 50, COMMA 1, LETT.B) DEL DLGS N.36/2023 - IMPEGNO DI...","“SERVIZIO DI SORVEGLIANZA SANITARIA PREVISTO DAL D-LGS 81/2008” - DETERMINA A CONTRARRE E AFFIDAMENTO DIRETTO, AI SE..."
1,Comune di Custonaci,6487,51,1.00,"ALIENAZIONE IMMOBILI DI PROPRIETÀ COMUNALE - DETERMINA A CONTRARRE (ART.17. COMMA 1 DEL D.LGS 31 MARZO 2023, N. 36) ...","ALIENAZIONE IMMOBILI DI PROPRIETÀ COMUNALE - DETERMINA A CONTRARRE (ART.17. COMMA 1 DEL D.LGS 31 MARZO 2023, N. 36) ..."
2,Comune di Alcamo,10393,1409,1.00,"BANDO DI CONCORSO PUBBLICO, PER ESAMI, PER L’ASSUNZIONE, A TEMPO PIENO E INDETERMINATO, DI N. 2 DIRIGENTI TECNICI – ...","BANDO DI CONCORSO PUBBLICO, PER ESAMI, PER L’ASSUNZIONE, A TEMPO PIENO E INDETERMINATO, DI N. 2 DIRIGENTI TECNICI – ..."
3,Comune di Giarre,11812,54,0.79,DETERMINA A CONTRARRE SEMPLIFICATA AFFIDAMENTO E IMPEGNO SPESA PER SERVIZIO CENTRO DIURNO ANZIANI DI A MEZZO AFFIDAM...,DETERMINA A CONTRARRE SEMPLIFICATA AFFIDAMENTO E IMPEGNO SPESA PER SERVIZIO CENTRO DIURNO ANZIANI DI A MEZZO AFFIDAM...
4,Comune di Marsala,20443,40,1.00,"DETERMINA A CONTRARRE, AI SENSI DELL’ART. 1, COMMA 2 DEL D.L. 76/2020, CONVERTITO IN LEGGE N. 120/2020 E MODIFICATO ...","DETERMINA A CONTRARRE, AI SENSI DELL’ART. 1, COMMA 2 DEL D.L. 76/2020, CONVERTITO IN LEGGE N. 120/2020 E MODIFICATO ..."
5,Comune di Giarre,23666,202,1.00,ATTO DI INDIRIZZO AL SINDACO E ALLA GIUNTA COMUNALE PER L’INDIZIONE DI UN CONCORSO DI IDEE O MANIFESTAZIONE DI INTER...,ATTO DI INDIRIZZO AL SINDACO E ALLA GIUNTA COMUNALE PER L’INDIZIONE DI UN CONCORSO DI IDEE O MANIFESTAZIONE DI INTER...
6,Comune di Ragusa,26206,264,1.00,"DETERMINAZIONE A CONTRARRE PER LA CONCLUSIONE DI UN ACCORDO QUADRO CON UN UNICO OPERATORE, AI SENSI DELL’ART.54, COM...","DETERMINAZIONE A CONTRARRE PER LA CONCLUSIONE DI UN ACCORDO QUADRO CON UN UNICO OPERATORE, AI SENSI DELL’ART.54, COM..."
7,Comune di Sperlinga,34796,2,0.51,Annullamento in autotutela della procedura negoziata per l'affidamento dei servizi tecnici per la direzione lavori e...,Determinazione a contrarre per affidamento dei servizi tecnici per la direzione lavori e sicurezza in fase di esecuz...


**Valutazione dei singoli 8, non solo del filtro che li ha lasciati passare:**

| proc_id | Ente | Giudizio |
|---|---|---|
| 692 | Palma di Montechiaro | ✅ già noto/validato in TAL-48 (affidamento diretto sorveglianza sanitaria) |
| 6487 | Custonaci | ✅ "approvazione bando di gara" esplicito |
| 10393 | Alcamo | ✅ concorso 2 dirigenti tecnici, annullamento reale (avviso di annullamento procedura concorsuale) |
| 11812 | Giarre | ✅ affidamento diretto con CIG, annullamento determina esplicito |
| 20443 | Marsala | ✅ "revoca in autotutela della determina a contrarre e della relativa gara" — il caso da manuale |
| 23666 | Giarre | ⚠️ un "atto di indirizzo" (fase preliminare politica) per un futuro concorso di idee, non ancora un bando vero — incluso perché la parola "concorso" c'è letteralmente, ma è un segnale debole |
| 26206 | Ragusa | ✅ accordo quadro, stessa determina ripubblicata identica |
| 34796 | Sperlinga | ✅ annullamento in autotutela esplicito di una procedura negoziata, riproposta 2 giorni dopo (jaccard più basso, 0.51, ma testo inequivocabile) |

**7/8 sono inequivocabili. 1/8 (Giarre 23666) è un segnale debole** (un indirizzo politico verso un futuro concorso, non un bando già pubblicato) — non lo definirei un falso positivo, ma nemmeno un caso "da manuale" come gli altri 7.

## Prova 4 — Cosa perdiamo: quanti dei "tolti" avevano comunque un CIG?

Il filtro a frasi guarda solo il *testo* dell'oggetto. Ma un CIG (Codice Identificativo Gara) è un segnale indipendente e molto più affidabile di dominio: se un procedimento tolto dal fix aveva un CIG associato, è un indizio che fosse **davvero** una gara, anche se la frase non la descrive con le formule che cerco ("determina a contrarre", "affidamento diretto"...). Questo è il modo più onesto che ho per stimare il costo in falsi negativi.

In [5]:
def cig_del_procedimento(pid: int) -> str | None:
    """CIG proprio del procedimento, o in mancanza il primo CIG tra i suoi atti."""
    p = con.execute("SELECT cig FROM procedimenti WHERE id=?", (pid,)).fetchone()
    if p["cig"]:
        return p["cig"]
    a = con.execute(
        "SELECT cig FROM atti WHERE procedimento_id=? AND cig IS NOT NULL AND cig != '' LIMIT 1", (pid,)
    ).fetchone()
    return a["cig"] if a else None


df_tolti["cig"] = df_tolti["proc_id"].map(cig_del_procedimento)
con_cig = df_tolti[df_tolti["cig"].notna()]
print(f"Procedimenti tolti CON un CIG associato (possibile falso negativo): {len(con_cig)} / {len(df_tolti)}")
con_cig[["proc_id", "ente", "cig", "oggetto"]]

Procedimenti tolti CON un CIG associato (possibile falso negativo): 3 / 59


,proc_id,ente,cig,oggetto
16,18582,Comune di Bagheria,Z9C3967EEC,"ABBONAMNETO A CORSI DI FORMAZIONE IN MATERIA FINANZIARIA, TRIBUTARIA E DEL PERSONALE PER GLI ENTI LOCALI – ANNO 2023..."
17,19564,Comune di Giarre,ZD939A2806,REVOCA DETERMINA DI IMPEGNO SPESA AREA STAFF N. 72 DEL 09/02/2023 CONSEQUENZIALE ANNULLAMENTO DEL CIG ZD939A2806 E R...
54,31392,Comune di Adrano,BA2D862925,"Annullamento in autotutela, ai sensi dell'art. 21-nonies della Legge n. 241/1990, della Determinazione n. 231 del 14..."


**Solo 3/59 avevano un CIG in colonna, e leggendoli nessuno è davvero un "bando riaperto" perso:**

- **Bagheria 18582** ("abbonamento a corsi di formazione"): il CIG appartiene a un contratto di abbonamento esistente, l'oggetto non descrive una gara ripubblicata.
- **Giarre 19564** ("revoca determina impegno spesa... consequenziale annullamento del CIG..."): è una correzione contabile su un contratto di servizio già in corso, non un nuovo bando.
- **Adrano 31392**: questo sì è un caso reale perso — "annullamento in autotutela" esplicito di un impegno per lavori di somma urgenza (art. 140 D.Lgs 36/2023), con CIG. È l'unico dei 59 dove il costo del filtro più stretto è concreto, non solo teorico.

**Limite di questo controllo, da dichiarare:** la colonna `cig` è popolata solo quando lo scraper l'ha estratta in un campo dedicato — **San Gregorio di Catania (24909)**, un secondo caso perso, non compare qui perché il suo CIG (`ZC137DE299`) è scritto dentro il testo dell'oggetto ma non è mai stato promosso alla colonna strutturata. Quindi "3/59 con CIG" è un limite inferiore, non il numero esatto di falsi negativi — il costo reale è probabilmente più vicino a 2 (Adrano + San Gregorio) che a 3, ma potrebbero essercene altri con CIG solo testuale che questo controllo non vede.

## Prova 5 — Il filtro generalizza, o è overfit ai casi che ho già letto?

Fin qui ho verificato il filtro solo sui procedimenti che il **matching Jaccard** aveva già selezionato come candidati riapertura. Ma il filtro di dominio (`_e_dominio_gara_appalti`) è stato scritto guardando quei casi specifici — rischio classico di overfitting.

Per un test più severo, lo applico a **tutti** i 555 procedimenti con `stato_finale IN ('annullato','revocato')` nel DB, non solo ai casi già noti, e ne leggo un campione casuale (seed fissato, riproducibile) sia tra quelli marcati dentro-dominio sia tra quelli fuori-dominio — per vedere se il giudizio regge su testi mai ispezionati durante lo sviluppo del fix.

In [6]:
import random

tutti = con.execute(
    "SELECT id, ente_id, oggetto FROM procedimenti WHERE stato_finale IN ('annullato','revocato')"
).fetchall()
print(f"Popolazione totale procedimenti annullati/revocati: {len(tutti)}")

gia_noti = proc_pre  # i procedimenti già ispezionati manualmente (82 flag originali)
candidati_nuovi = [r for r in tutti if r["id"] not in gia_noti]
print(f"Mai ispezionati finora in questo lavoro: {len(candidati_nuovi)}")

random.seed(59)
campione = random.sample(candidati_nuovi, 40)

righe = []
for r in campione:
    dentro = riap_post._e_dominio_gara_appalti(r["oggetto"] or "")
    righe.append({"proc_id": r["id"], "ente": nomi_enti.get(r["ente_id"], "?"), "in_dominio": dentro, "oggetto": r["oggetto"]})
df_campione = pd.DataFrame(righe)
print(f"Nel campione casuale di {len(df_campione)}: {df_campione['in_dominio'].sum()} classificati IN dominio, {(~df_campione['in_dominio']).sum()} FUORI dominio")

Popolazione totale procedimenti annullati/revocati: 555
Mai ispezionati finora in questo lavoro: 488
Nel campione casuale di 40: 9 classificati IN dominio, 31 FUORI dominio


In [7]:
# Tutti quelli che il filtro marca IN dominio, nel campione mai visto prima
df_campione[df_campione["in_dominio"]][["proc_id", "ente", "oggetto"]]

,proc_id,ente,oggetto
0,7427,Comune di Villabate,"DETERMINA A CONTRARRE PER L’AFFIDAMENTO DELL’INCARICO DI N. 1 OPERAIO QUALIFICATO, DA AVVIARE NEL CANTIERE DI LAVORO..."
4,35499,Comune di Vittoria,"Determina n. 3217 del 25/06/2026. Modifica allegato A) ""Bando di concorso pubblico per il rilascio di "" N. 8 (otto) ..."
6,653,Comune di Palma di Montechiaro,APPROVAZIONE AVVISO DI SELEZIONE E MODELLO ISTANZA PER SELEZIONE INTERNA PER PROGRESSIONE TRA LE AREE (VERTICALI) “I...
8,31488,Comune di Milazzo,DETERMINAZIONE DI REVOCA DELL’AFFIDAMENTO DISPOSTO ALLA OFFICINE METALLICHE DOPPIA C. S.R.L. CON DETERMINAZIONE DIRI...
26,11006,Comune di Ragusa,"AFFIDAMENTO DEI SERVIZI AUSILIARI DEL COMPLESSO CASTELLO DI DONNAFUGATA FINALIZZATI ALLO SVILUPPO ECONOMICO, ALLA PR..."
31,33794,Comune di Piraino,GESTIONE ESTERNALIZZATA DEL SERVIZIO EDUCATIVO DELL'ASILO NIDO COMUNALE DAL 01 SETTEMBRE 2026 AL 31 LUGLIO 2027 - DE...
33,4888,Comune di Nicolosi,EMERGENZA ETNA_CADUTA CENERE VULCANICA_14 AGOSTO 2023. SERVIZIO AGGIUNTIVO PER LO SPAZZAMENTO E LA RACCOLTA DELLA CE...
38,10368,Comune di Castel di Iudica,SELEZIONE COMPARATIVA PROGRESSIONI TRA LE AREE ANNO 2024 CCNL 16.11.2022. AVVIO E APPROVAZIONE BANDO DI SELEZIONE.
39,3690,Comune di Acireale,REVOCA DETERMINA DIRIGENZIALE N° 266 DEL 05/06/2026 E AFFIDAMENTO DELLA FORNITURA DI DUE GOMMONI NAUTICI DI SALVATAG...


In [8]:
# Un sotto-campione di quelli marcati FUORI dominio, per controllare falsi negativi
df_campione[~df_campione["in_dominio"]][["proc_id", "ente", "oggetto"]].head(20)

,proc_id,ente,oggetto
1,3024,Comune di Giarre,TRASFERIMENTO SOMME DA PARTE DELLO STATO PER ASSISTENZA ALL'AUTONOMIA E ALLA COMUNICAZIONE PER GLI ALUNNI CON DISABI...
2,35016,Comune di Catania,REVOCA PROVVEDIMENTO N.11.2024 DEL 16.06.2026. LIQUIDAZIONE FATTURA N.9 DEL 07.07.2026 (PROT.N.319096 DEL 07.07.2026...
3,23387,Comune di Acireale,INTERVENTI PREVISTI NELL’AMBITO DELLA SOTTOMISURA 19.2 DEL PSR SICILIA 2014-2022 – STRATEGIA DI SVILUPPO LOCALE DI T...
5,20089,Comune di Trapani,"ISTITUZIONE DIVIETI DI CIRCOLAZIONE, DI SOSTA CON RIMOZIONE FORZATA E SENSI UNICI DI MARCIA, IN OCCASIONE DEGLI SPET..."
7,6349,Comune di Gravina di Catania,"FSC 2014-2020. ""PATTO PER IL SUD"" DELIBERA DI GIUNTA N. 70 DEL 27.02.2020. TEATRI IN SICILIA. INTERVENTO . APPROVAZI..."
9,10169,Comune di Castel di Iudica,REVOCA DETERMINA N. 1179 DEL 09.10.2023 E CONTESTUALE IMPEGNO DI SPESA PER ACQUISTO DISSUASORI DI SOSTA DA ALLOCATE ...
10,25095,Comune di Sant'Agata li Battiati,RINVIO SEDUTA PER MANCANZA DEL NUMERO LEGALE.
11,23801,Comune di Giarre,"COMUNE DI GIARRE C/ ING. G.R.LEONARDI - LIQUIDAZIONE FATTURA N° 36 DEL 12/06/2025 DI EURO 4.520,18 IN ACCONTO PER L'..."
12,22598,Comune di Caltanissetta,REVOCA ASSESSORE COMUNALE MATILDE DANIELA LOREDANA FALCONE
13,335,Comune di Caltanissetta,"C.G.A .- PERSONA (1), IN ATTI ALLEGATI MEGLIO IDENTIFICATA + 1 C/COMUNE - RICORSO IN APPELLO N. 497/2024 RG PER L'AN..."


**Il filtro regge nel complesso, ma il campione cieco ha trovato due lacune reali che i 9 fascicoli originari non avevano fatto emergere:**

**1) I 7 marcati IN dominio: 6/7 palesemente corretti** (bando di concorso pubblico per licenze, affidamento servizi ausiliari Castello di Donnafugata, gestione esternalizzata asilo nido, servizio spazzamento emergenza Etna, revoca+affidamento fornitura gommoni di salvataggio). **1/7 è un caso di confine**: Villabate 7427, "determina a contrarre per l'affidamento dell'incarico di un operaio qualificato... cantiere di lavoro" — un "cantiere di lavoro" è una misura di politica attiva del lavoro (collocamento temporaneo sussidiato), non un vero appalto competitivo, anche se usa lo stesso linguaggio contrattuale. Non è un errore del filtro, ma un promemoria che "determina a contrarre" cattura anche atti a cavallo tra procurement e politiche sociali.

**2) Tra i 33 fuori dominio, trovate due lacune concrete (falsi negativi):**

- **Milazzo 31488** — `"DETERMINAZIONE DI REVOCA DELL'AFFIDAMENTO DISPOSTO ALLA OFFICINE METALLICHE DOPPIA C. S.R.L...."` — questo **è** un affidamento revocato, testualmente. Il filtro (nella sua prima versione) lo perdeva per un motivo puramente meccanico: la regex cercava `affidamento\s+(?:diretto|dei|del|della|delle)` con uno spazio dopo "affidamento", ma qui c'è l'elisione `dell'affidamento` → `AFFIDAMENTO DISPOSTO...`, senza nessuna delle parole che cerca subito dopo.
- **Palma di Montechiaro 653 vs. Castel di Iudica 10368**: stessa fattispecie (concorso interno per progressione tra le aree), stesso tipo di provvedimento — uno diceva `"BANDO di selezione"` (→ dentro dominio) e l'altro `"AVVISO di selezione"` (→ fuori dominio). Incoerenza reale: il filtro premiava la parola scelta dall'ente più che la sostanza dell'atto.

**Entrambe corrette nello stesso giro di lavoro — verificato in Prova 8 più sotto.**

In [9]:
# Prova 5b — le due lacune trovate sopra sono riproducibili sul codice ATTUALE?
casi_limite = [
    ("Milazzo 31488 (REVOCA DELL'AFFIDAMENTO, elisione)",
     "DETERMINAZIONE DI REVOCA DELL'AFFIDAMENTO DISPOSTO ALLA OFFICINE METALLICHE DOPPIA C. S.R.L. CON DETERMINAZIONE"),
    ("Palma 653 (AVVISO di selezione interna)",
     "APPROVAZIONE AVVISO DI SELEZIONE E MODELLO ISTANZA PER SELEZIONE INTERNA PER PROGRESSIONE TRA LE AREE"),
    ("Castel di Iudica 10368 (BANDO di selezione, stessa fattispecie)",
     "SELEZIONE COMPARATIVA PROGRESSIONI TRA LE AREE ANNO 2024. AVVIO E APPROVAZIONE BANDO DI SELEZIONE"),
]
pd.DataFrame([
    {"caso": nome, "in_dominio (codice attuale)": riap_post._e_dominio_gara_appalti(testo)}
    for nome, testo in casi_limite
])

,caso,in_dominio (codice attuale)
0,"Milazzo 31488 (REVOCA DELL'AFFIDAMENTO, elisione)",True
1,Palma 653 (AVVISO di selezione interna),True
2,"Castel di Iudica 10368 (BANDO di selezione, stessa fattispecie)",True


## Prova 8 — Le due lacune sono state corrette: verifica di non-regressione

Dopo aver trovato le due lacune sopra, la regex è stata estesa (`disposto` come possibile parola dopo "affidamento", `avviso di selezione` come frase equivalente a `bando di selezione`). Non basta verificare che i due casi noti ora passino: bisogna controllare che l'estensione non abbia **riaperto la porta a nuovi falsi positivi** altrove. Ricalcolo la classificazione di dominio su tutti i 555 procedimenti annullati/revocati con la regex del primo giro (commit `3e459e4`, quello con cui è stato scritto questo notebook la prima volta) e quella attuale, e guardo ogni singola differenza.

In [10]:
sorgente_v2 = subprocess.run(
    ["git", "show", f"{POST_FIX_COMMIT}:src/talia/modulo2_scraping/red_flags/riapertura_revoca.py"],
    cwd=REPO, capture_output=True, text=True, check=True,
).stdout
tmp_path = Path("/tmp/_tal59_riap_v2.py")
tmp_path.write_text(sorgente_v2)
spec = importlib.util.spec_from_file_location("riap_v2", tmp_path)
riap_v2 = importlib.util.module_from_spec(spec)
sys.modules["riap_v2"] = riap_v2
spec.loader.exec_module(riap_v2)

tutti_555 = con.execute(
    "SELECT id, ente_id, oggetto FROM procedimenti WHERE stato_finale IN ('annullato','revocato')"
).fetchall()

cambiati = []
for r in tutti_555:
    prima = riap_v2._e_dominio_gara_appalti(r["oggetto"] or "")
    dopo = riap_post._e_dominio_gara_appalti(r["oggetto"] or "")
    if prima != dopo:
        cambiati.append({"proc_id": r["id"], "ente": nomi_enti.get(r["ente_id"], "?"), "prima": prima, "dopo": dopo, "oggetto": r["oggetto"]})

df_cambiati = pd.DataFrame(cambiati)
print(f"Procedimenti la cui classificazione cambia: {len(df_cambiati)} / {len(tutti_555)}")
if len(df_cambiati):
    print(f"Di questi, nuovi FALSI (dovrebbero essere 0): {(~df_cambiati['dopo']).sum()}")
df_cambiati

Procedimenti la cui classificazione cambia: 5 / 555
Di questi, nuovi FALSI (dovrebbero essere 0): 0


,proc_id,ente,prima,dopo,oggetto
0,653,Comune di Palma di Montechiaro,False,True,APPROVAZIONE AVVISO DI SELEZIONE E MODELLO ISTANZA PER SELEZIONE INTERNA PER PROGRESSIONE TRA LE AREE (VERTICALI) “I...
1,654,Comune di Palma di Montechiaro,False,True,APPROVAZIONE AVVISO DI SELEZIONE E MODELLO ISTANZA PER SELEZIONE INTERNA PER PROGRESSIONE TRA LE AREE (VERTICALI) “I...
2,655,Comune di Palma di Montechiaro,False,True,APPROVAZIONE AVVISO DI SELEZIONE E MODELLO ISTANZA PER SELEZIONE INTERNA PER PROGRESSIONE TRA LE AREE (VERTICALI) “I...
3,7045,Comune di Bagheria,False,True,AVVISO DI SELEZIONE INTERNA PER CONFERIMENTO DI INCARICHI DI ELEVATA QUALIFICAZIONE.
4,31488,Comune di Milazzo,False,True,DETERMINAZIONE DI REVOCA DELL’AFFIDAMENTO DISPOSTO ALLA OFFICINE METALLICHE DOPPIA C. S.R.L. CON DETERMINAZIONE DIRI...


**5 procedimenti cambiano classificazione, tutti nella direzione attesa (falso→vero): nessuna regressione.** 3 sono altre occorrenze dello stesso "avviso di selezione interna per progressione tra le aree" di Palma di Montechiaro (stesso processo HR ripetuto), 1 è un caso analogo in un altro ente, 1 è Milazzo. Nessun procedimento passa da dentro a fuori dominio — l'estensione è stata puramente additiva sui due casi diagnosticati, non ha toccato nient'altro.

## Prova 6 — Nessuna regressione sugli altri red flag

Il fix tocca solo `engine/catena.py::classifica_ruolo` e `red_flags/riapertura_revoca.py`. Verifico due cose indipendenti: (a) il diff del branch non tocca nessun altro modulo di red flag, (b) gli altri red flag deterministici girano ancora senza eccezioni sullo stesso DB e producono conteggi ragionevoli (non necessariamente identici — `classifica_ruolo` è condiviso e in teoria potrebbe influenzarli se costruissero catene da zero in questa sessione, cosa che non fanno: leggono `procedimenti`/`atti` già popolati).

In [11]:
file_toccati = subprocess.run(
    ["git", "diff", "--name-only", PRE_FIX_COMMIT],
    cwd=REPO, capture_output=True, text=True, check=True,
).stdout.splitlines()
print("File modificati rispetto a prima del fix (inclusi commit non ancora fatti in questo giro):")
for f in file_toccati:
    print(" -", f)

assert "src/talia/modulo2_scraping/red_flags/concentrazione.py" not in file_toccati
assert "src/talia/modulo2_scraping/red_flags/frazionamento.py" not in file_toccati
assert "src/talia/modulo2_scraping/red_flags/tempi_anomali.py" not in file_toccati
assert "src/talia/modulo2_scraping/red_flags/catena_revoca.py" not in file_toccati
print("\nConfermato: nessun altro modulo red_flags nel diff.")

File modificati rispetto a prima del fix (inclusi commit non ancora fatti in questo giro):
 - HANDOFF.md
 - docs/cards/BOARD.md
 - docs/cards/TAL-59.md
 - src/talia/engine/catena.py
 - src/talia/modulo2_scraping/red_flags/riapertura_revoca.py
 - tests/red_flags/test_riapertura_revoca.py
 - tests/test_catena.py

Confermato: nessun altro modulo red_flags nel diff.


In [12]:
from talia.modulo2_scraping.red_flags import catena_revoca, concentrazione, frazionamento, tempi_anomali

esiti = {}
for nome, funzione in [
    ("concentrazione", concentrazione.rileva_concentrazione),
    ("frazionamento", frazionamento.rileva_frazionamento),
    ("tempi_anomali", tempi_anomali.rileva_tempi_anomali),
    ("revoche_in_catena", catena_revoca.rileva_revoche_in_catena),
]:
    try:
        r = funzione(nuova_connessione())
        esiti[nome] = ("OK", len(r))
    except Exception as e:
        esiti[nome] = ("ERRORE", str(e))

pd.DataFrame(esiti).T.rename(columns={0: "esito", 1: "conteggio/errore"})

,esito,conteggio/errore
concentrazione,OK,501
frazionamento,OK,0
tempi_anomali,OK,1
revoche_in_catena,OK,48


**Nessuna eccezione**, tutti e 4 gli altri red flag girano puliti sullo stesso DB. `frazionamento=0` non è una regressione: è così anche in run storici precedenti non collegati a questo fix (vedi `HANDOFF.md`, run del 2026-07-20), quindi è un dato pre-esistente del dataset/algoritmo, non un effetto collaterale di TAL-59.

## Prova 7 — Il Fix 1 (tag `[annullato]`) non è retroattivo: quanto resta da bonificare?

`classifica_ruolo()` viene chiamata solo quando un atto viene **inserito/riclassificato** in catena — non quando si interroga il DB. I dati già scritti in `talia.db` restano con il `ruolo_in_catena` sbagliato finché non gira un nuovo run scraper (o un backfill esplicito, non eseguito qui). Quanto è grande il debito residuo?

In [13]:
tag_atti = con.execute(
    "SELECT id, ente_id, oggetto, ruolo_in_catena FROM atti WHERE oggetto LIKE '[%'"
).fetchall()
df_tag = pd.DataFrame([dict(r) for r in tag_atti])
print(f"Atti nel DB con oggetto che inizia per '[': {len(df_tag)}")
print(df_tag["ruolo_in_catena"].value_counts(dropna=False))

# Verifica diretta: la NUOVA classifica_ruolo() su questi stessi testi darebbe
# davvero un ruolo diverso da quello salvato (cioè il fix li correggerebbe se
# rilanciassimo la ricostruzione catene)?
df_tag["ruolo_con_fix"] = df_tag["oggetto"].apply(lambda o: catena_post.classifica_ruolo(oggetto=o))
cambierebbero = df_tag[df_tag["ruolo_in_catena"] != df_tag["ruolo_con_fix"]]
print(f"\nDi questi, il ruolo cambierebbe con Fix 1 attivo: {len(cambierebbero)} / {len(df_tag)}")
cambierebbero[["id", "oggetto", "ruolo_in_catena", "ruolo_con_fix"]].head(15)

Atti nel DB con oggetto che inizia per '[': 177
ruolo_in_catena
annullamento    120
NaN              31
liquidazione     24
revoca            2
Name: count, dtype: int64

Di questi, il ruolo cambierebbe con Fix 1 attivo: 151 / 177


,id,oggetto,ruolo_in_catena,ruolo_con_fix
0,298,[annullato] TASSI DI ASSENZA DICEMBRE 2024,annullamento,altro
1,2037,[annullato] COSTO PERSONALE NON A TEMPO INDETERMINATO - II TRIM. 2023,annullamento,altro
2,2479,[p: 16621-2026] Manifesto di sgombero relativo al poligono di Drasi AG 2 semestre 2026.,NaN,altro
3,2480,"[p: 16675-2026] Poligono di Drasi. Provvedimenti da attuare nel corso di esercitazioni militari, per la salvaguardia...",NaN,altro
4,3670,"[annullato] BANDO PUBBLICO, PER L'ASSEGNAZIONE E LA CESSIONE, IN DIRITTO DI PROPRIETA’, DI N. 10 (DIECI) LOTTI INSER...",annullamento,avvio
5,7434,"[annullato] ETERMINA A CONTRARRE E CONTESTUALE AFFIDAMENTO AL TIRO A SEGNO NAZIONALE, SEZIONE DI PALERMO, PER L'ESER...",NaN,aggiudicazione
6,7443,"[annullato] PIAZZA MARIA- E.Q. ""PATRIMONIO E BENI CONFISCATI""- ANNO 2025",annullamento,altro
7,7585,"[annullato] AFFIDAMENTO DIRETTO EX ART. 50, COMMA 1, LETTERA A) D.LGS.36/2023 INFERIORE A € 5.000,00 PER LA FORNITUR...",NaN,aggiudicazione
8,7774,"[annullato] DETERMINA A CONTRARRE E AFFIDAMENTO DIRETTO MEDIANTE O.D.A. SUL ME.PA., PER I SERVIZI AGGIUNTIVI E MANUT...",annullamento,aggiudicazione
9,7909,"[annullato] PROROGA CONFERIMENTO INCARICO DI E.Q. ""SERVIZI A RETE"" GEOM. FRANCESCO SERGIO PALUMBO- ANNO 2024",NaN,proroga


**151/177 atti con tag di stato tra parentesi quadre hanno oggi un `ruolo_in_catena` sbagliato che il Fix 1 correggerebbe** — numero che coincide con la stima indipendente del sotto-agente che avevo lanciato durante l'investigazione iniziale (anche lui aveva trovato 151), un controllo incrociato che rassicura sulla diagnosi. **Ma restano sbagliati finché qualcuno non rilancia la ricostruzione delle catene** (nuovo run scraper, o un backfill dedicato) — non l'ho fatto qui perché scrive sul DB reale e va deciso esplicitamente, non lanciato di sorpresa dentro una verifica.

Nota anche i 26 casi con tag diverso da `[annullato]` (es. `[p: 16621-2026]`, un riferimento a un permesso di poligono militare) che restano `ruolo=None`/`altro` sia prima che dopo — non tutti i tag tra parentesi quadre sono lo stesso bug, ma la stragrande maggioranza (120/177) lo è.

## Conclusione critica

**Cosa il fix risolve bene, verificato non solo dichiarato:**
- Riduzione confermata dal vivo: **67 → 8 procedimenti (-88,1%)**, non solo il numero di righe.
- I 59 tolti sono stati letti **tutti**, non a campione: nessuno mi sembra un'esclusione sbagliata. Ha anche fatto emergere un pattern non visto prima (17 casi dallo stesso ente, Sant'Agata li Battiati) che i 9 campioni originari di TAL-12 non coprivano.
- Il filtro **generalizza** ragionevolmente su un campione cieco di 40 procedimenti mai ispezionati durante lo sviluppo: 6/7 classificazioni positive erano già corrette; le 2 lacune reali trovate (elisione "dell'affidamento", incoerenza bando/avviso di selezione) sono state corrette nello stesso giro e la correzione è stata verificata **non regressiva** su tutti i 555 procedimenti (Prova 8): solo 5 cambiano classificazione, tutti nella direzione giusta.
- Nessuna regressione sugli altri 4 red flag deterministici.

**Cosa resta scoperto — non nascosto, da decidere:**
1. **Costo reale in falsi negativi stimato in 2 casi su 59** (Adrano, San Gregorio di Catania) — piccolo ma non nullo, e la sua stima stessa ha un limite dichiarato (colonna CIG non sempre popolata).
2. **Il Fix 1 (tag `[annullato]`) non ha ancora effetto sui dati esistenti** — 151 atti restano classificati male finché non gira un backfill, decisione che spetta a Dom, non presa qui.
3. **Bug 2b (Jaccard che lega atti scorrelati anche dentro lo stesso dominio) resta esplicitamente fuori scope**, come già scritto nella card — questa verifica non lo tocca, ed è dove il rischio residuo più grande vive ancora.
4. Un solo caso residuo tra gli 8 finali (Giarre 23666) è un segnale debole (un indirizzo politico, non ancora un bando).

**Giudizio:** il fix fa quello che promette, verificato con dati reali end-to-end (non solo test sintetici) e messo sotto pressione con un campione cieco che ha effettivamente trovato — e permesso di correggere — due lacune concrete. I punti 1-4 sono limiti noti e documentati, da programmare come passi successivi con Dom, non difetti nascosti di questo giro di lavoro.